# Population-size branch check on the Colab T4

Diagnostic side-by-side, not a sweep for the paper. `runs/ks-correctness/ks_n200_top/`
(n_agents=200, num_envs=32, kappas=null) settled into a persistent bimodal,
moderate-high-Gini (~0.42) wealth distribution. `runs/ks-wealth-lognormal-random/`'s
sigma=0 cells (n_agents=500, num_envs=8, also kappas≡1) instead landed in a much
lower-Gini regime (~0.02-0.09) in all 5 seeds tried. This notebook runs several
seeds at both population/batch-width combinations, with the full
Gini-over-training trajectory recorded (`diag.n_snapshots=12`, not just the final
checkpoint), to check whether population size is a real confound or the sigma=0
seeds just happened to sample the low-Gini basin by chance. Full design writeup:
`runs/ks-population-branch-check/README.md`.

Default config: 6 seeds x 2 arms = 12 runs. **Time the first run before assuming
the rest fit in one Colab session.** Each run's outputs are written as soon as it
finishes, so a disconnect partway through only loses the run in progress.

In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps

In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L

In [ ]:
# Mount Drive BEFORE the run so results are saved as soon as each of the 12
# runs finishes (a Colab disconnect then loses at most the run in progress).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-population-branch-check'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did any run finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")

## Run both arms

Runs `runs/ks-population-branch-check/config.yaml` as-is: 6 seeds,
`n_snapshots=12`, `device=gpu`. Pass dotlist overrides after the script path to
change any of these -- e.g. a quick CPU smoke test with 2 seeds and fewer
snapshots:
`!python runs/ks-population-branch-check/run_branch_check.py "seeds=[0,1]" device=cpu n_snapshots=3 sim_steps=200`

In [ ]:
!python runs/ks-population-branch-check/run_branch_check.py

In [ ]:
save_results('ks-population-branch-check')

## Results: final-snapshot Gini per arm/seed

In [ ]:
import json, glob
import pandas as pd

rows = []
for d in sorted(glob.glob('runs/ks-population-branch-check/results/ks/*')):
    run_name = d.split('/')[-1]
    arm, seed = run_name.rsplit('_seed', 1)
    diag = json.load(open(f'{d}/diagnostics.json'))
    f = diag['final']
    rows.append({
        'arm': arm, 'seed': int(seed),
        'capital_gini': f['distributional']['capital_gini'],
        'top_0.1_share': f['distributional']['top_0.1_share'],
        'K_mean': f['economic']['steady_state']['K_mean'],
        'resource_mean_rel': f['economic']['resource']['resource_mean_rel'],
    })
df = pd.DataFrame(rows)
display(df)
display(df.groupby('arm')['capital_gini'].describe())

## Per-run figures

`distributional.png` (Gini/top-share vs. training steps) and
`ks_wealth_heatmap.png` (stationary wealth distribution through training) for
every run -- the same views `ks_n200_top` has, so each run can be compared
directly against it.

In [ ]:
from IPython.display import Image, display

for d in sorted(glob.glob('runs/ks-population-branch-check/results/ks/*')):
    run_name = d.split('/')[-1]
    print(f"== {run_name} ==")
    display(Image(f'{d}/figures/distributional.png'))
    display(Image(f'{d}/figures/ks_wealth_heatmap.png'))